In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from scipy.stats import ks_2samp
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from pyspark.sql import functions as F

catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]

TRAIN_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_train"
TEST_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_test"
PREDICTIONS_TABLE = f"{catalog_name}.{schema_name}.fraud_predictions_test"

reference_pdf = spark.table(TRAIN_TABLE).toPandas()
current_pdf = spark.table(TEST_TABLE).toPandas()
predictions_pdf = spark.table(PREDICTIONS_TABLE).toPandas()

feature_columns = [c for c in reference_pdf.columns 
                   if c != "Class"
]

reference_pdf = reference_pdf[feature_columns].copy()
current_pdf = current_pdf[feature_columns].copy()

print("Reference rows:", len(reference_pdf))
print("Current batch rows:", len(current_pdf))
print("Features monitored:", len(feature_columns))
print("Prediction rows:", len(predictions_pdf))


Reference rows: 227845
Current batch rows: 56962
Features monitored: 30
Prediction rows: 56962


In [0]:
PSI_THRESHOLD = 0.20
KS_PVALUE_THRESHOLD = 0.05
def calculate_psi(reference, current, bins=10):
   ref = pd.Series(reference).dropna().to_numpy()
   cur = pd.Series(current).dropna().to_numpy()
   if len(ref) == 0 or len(cur) == 0:
       return np.nan
   # Build bins using reference-data quantiles.
   edges = np.unique(
       np.quantile(ref, np.linspace(0, 1, bins + 1))
   )
   if len(edges) < 3:
       return 0.0
   edges[0] = -np.inf
   edges[-1] = np.inf
   ref_counts = np.histogram(ref, bins=edges)[0]
   cur_counts = np.histogram(cur, bins=edges)[0]
   # Small smoothing value prevents division by zero.
   epsilon = 1e-6
   ref_pct = ref_counts / max(ref_counts.sum(), 1)
   cur_pct = cur_counts / max(cur_counts.sum(), 1)
   ref_pct = np.clip(ref_pct, epsilon, None)
   cur_pct = np.clip(cur_pct, epsilon, None)
   return float(
       np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
   )

def calculate_drift(reference, current, batch_name):
   results = []
   for feature in feature_columns:
       ref = reference[feature].dropna()
       cur = current[feature].dropna()
       if len(ref) == 0 or len(cur) == 0:
           psi = np.nan
           ks_stat = np.nan
           ks_pvalue = np.nan
       else:
           psi = calculate_psi(ref, cur)
           ks_result = ks_2samp(ref, cur)
           ks_stat = float(ks_result.statistic)
           ks_pvalue = float(ks_result.pvalue)
       results.append({
           "batch_name": batch_name,
           "feature": feature,
           "psi": psi,
           "ks_statistic": ks_stat,
           "ks_pvalue": ks_pvalue,
           "psi_alert": bool(
               pd.notna(psi) and psi > PSI_THRESHOLD
           ),
           "ks_alert": bool(
               pd.notna(ks_pvalue)
               and ks_pvalue < KS_PVALUE_THRESHOLD
           )
       })
   return pd.DataFrame(results)

normal_drift_pdf = calculate_drift(
   reference_pdf,
   current_pdf,
   "normal_test_batch"
)
display(
   normal_drift_pdf.sort_values("psi", ascending=False)
)

batch_name,feature,psi,ks_statistic,ks_pvalue,psi_alert,ks_alert
normal_test_batch,V12,4.2917669373032045E-4,0.0073519413911564135,0.014437977483656033,false,true
normal_test_batch,V8,3.938037737017198E-4,0.005475900394579181,0.12956176861205237,false,false
normal_test_batch,V6,2.920753148549618E-4,0.0033491449647025107,0.6849361086533563,false,false
normal_test_batch,V18,2.5224723351036164E-4,0.003002849582799727,0.8046033937579904,false,false
normal_test_batch,V19,2.4294288564008176E-4,0.005707163360761602,0.10235383082688176,false,false
normal_test_batch,V11,2.1824510479591635E-4,0.004513102277206582,0.310380937845245,false,false
normal_test_batch,V10,2.1088264789987732E-4,0.0025308301855052617,0.9315368633477643,false,false
normal_test_batch,V21,2.093433496857873E-4,0.004050494594297627,0.44217884020311415,false,false
normal_test_batch,V9,2.075017294280909E-4,0.005193302863824245,0.17050911566365934,false,false
normal_test_batch,V16,1.9834646686405156E-4,0.006687143192632639,0.03381553581852392,false,true


In [0]:
simulated_pdf = current_pdf.copy()

simulated_pdf["Amount"] = simulated_pdf["Amount"] * 1.5
simulated_pdf["V14"] = simulated_pdf["V14"] + 0.5

simulated_drift_pdf = calculate_drift(
    reference_pdf, 
    simulated_pdf, 
    "simulated_drift_batch"
)

display(simulated_drift_pdf.sort_values("psi", ascending=False))

print(
    "Features with PSI alerts:",
    int(simulated_drift_pdf["psi_alert"].sum())
)

print(
    "Features with KS alerts:",
    int(simulated_drift_pdf["ks_alert"].sum())
)



batch_name,feature,psi,ks_statistic,ks_pvalue,psi_alert,ks_alert
simulated_drift_batch,Time,null,0.00538215394051389,0.14215291702018173,false,false
simulated_drift_batch,V1,null,0.0029963353511769375,0.8067124865629076,false,false
simulated_drift_batch,V2,null,0.0025091202151374947,0.9358159529122658,false,false
simulated_drift_batch,V3,null,0.0018852499911875265,0.9968513396611319,false,false
simulated_drift_batch,V4,null,0.0029651984874817794,0.8166893480363864,false,false
simulated_drift_batch,V5,null,0.002855471227476425,0.850312005579963,false,false
simulated_drift_batch,V6,null,0.0033491449647025107,0.6849361086533563,false,false
simulated_drift_batch,V7,null,0.002552760674305876,0.9270567035014528,false,false
simulated_drift_batch,V8,null,0.005475900394579181,0.12956176861205237,false,false
simulated_drift_batch,V9,null,0.005193302863824245,0.17050911566365934,false,false


Features with PSI alerts: 0
Features with KS alerts: 4


In [0]:
y_true = predictions_pdf["actual_class"].astype(int)
y_pred = predictions_pdf["predicted_class"].astype(int)
y_prob = predictions_pdf["fraud_probability"].astype(float)
performance_metrics = {
   "precision": precision_score(
       y_true, y_pred, zero_division=0
   ),
   "recall": recall_score(
       y_true, y_pred, zero_division=0
   ),
   "f1_score": f1_score(
       y_true, y_pred, zero_division=0
   ),
   "roc_auc": roc_auc_score(y_true, y_prob),
   "average_precision": average_precision_score(y_true, y_prob)
}
performance_pdf = pd.DataFrame([
   {
       "metric_name": name,
       "metric_value": float(value),
       "model_run_id": str(predictions_pdf["model_run_id"].iloc[0]),
       "measured_at_utc": datetime.now(timezone.utc).isoformat()
   }
   for name, value in performance_metrics.items()
])
display(performance_pdf)

metric_name,metric_value,model_run_id,measured_at_utc
precision,0.06097560975609756,ec060bf2a1fb44d29c250a12965238ee,2026-10-11T09:44:01.454811+00:00
recall,0.9183673469387755,ec060bf2a1fb44d29c250a12965238ee,2026-10-11T09:44:01.454862+00:00
f1_score,0.11435832274459974,ec060bf2a1fb44d29c250a12965238ee,2026-10-11T09:44:01.454877+00:00
roc_auc,0.9721687370080279,ec060bf2a1fb44d29c250a12965238ee,2026-10-11T09:44:01.454888+00:00
average_precision,0.7159122424484009,ec060bf2a1fb44d29c250a12965238ee,2026-10-11T09:44:01.454898+00:00


In [0]:
NORMAL_DRIFT_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_drift_normal"
)
SIMULATED_DRIFT_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_drift_simulated"
)
PERFORMANCE_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_model_performance"
)
(spark.createDataFrame(normal_drift_pdf)
   .write.format("delta")
   .mode("overwrite")
   .saveAsTable(NORMAL_DRIFT_TABLE))
(spark.createDataFrame(simulated_drift_pdf)
   .write.format("delta")
   .mode("overwrite")
   .saveAsTable(SIMULATED_DRIFT_TABLE))
(spark.createDataFrame(performance_pdf)
   .write.format("delta")
   .mode("append")
   .saveAsTable(PERFORMANCE_TABLE))
print("Saved normal drift:", NORMAL_DRIFT_TABLE)
print("Saved simulated drift:", SIMULATED_DRIFT_TABLE)
print("Saved performance metrics:", PERFORMANCE_TABLE)

Saved normal drift: workspace.default.fraud_drift_normal
Saved simulated drift: workspace.default.fraud_drift_simulated
Saved performance metrics: workspace.default.fraud_model_performance


In [0]:
print("NORMAL TEST BATCH")
print("Features monitored:", len(normal_drift_pdf))
print("PSI alerts:", int(normal_drift_pdf["psi_alert"].sum()))
print("KS alerts:", int(normal_drift_pdf["ks_alert"].sum()))
print("\nSIMULATED DRIFT BATCH")
print("Features monitored:", len(simulated_drift_pdf))
print("PSI alerts:", int(simulated_drift_pdf["psi_alert"].sum()))
print("KS alerts:", int(simulated_drift_pdf["ks_alert"].sum()))
print("\nMODEL PERFORMANCE")
for name, value in performance_metrics.items():
   print(f"{name}: {value:.4f}")
print("\nTop features by simulated PSI:")
display(
   spark.table(SIMULATED_DRIFT_TABLE)
   .orderBy(F.desc("psi"))
   .select(
       "feature", "psi", "ks_statistic",
       "ks_pvalue", "psi_alert", "ks_alert"
   )
   .limit(10)
)

NORMAL TEST BATCH
Features monitored: 30
PSI alerts: 0
KS alerts: 2

SIMULATED DRIFT BATCH
Features monitored: 30
PSI alerts: 0
KS alerts: 4

MODEL PERFORMANCE
precision: 0.0610
recall: 0.9184
f1_score: 0.1144
roc_auc: 0.9722
average_precision: 0.7159

Top features by simulated PSI:


feature,psi,ks_statistic,ks_pvalue,psi_alert,ks_alert
V8,null,0.005475900394579181,0.12956176861205237,false,false
V5,null,0.002855471227476425,0.850312005579963,false,false
V6,null,0.0033491449647025107,0.6849361086533563,false,false
V7,null,0.002552760674305876,0.9270567035014528,false,false
V1,null,0.0029963353511769375,0.8067124865629076,false,false
V4,null,0.0029651984874817794,0.8166893480363864,false,false
Time,null,0.00538215394051389,0.14215291702018173,false,false
V3,null,0.0018852499911875265,0.9968513396611319,false,false
V2,null,0.0025091202151374947,0.9358159529122658,false,false
V9,null,0.005193302863824245,0.17050911566365934,false,false
